IMPORTANT

If you haven't already, create a .venv and install dependencies

After creating a .venv, run this in your code editor's terminal inside the .venv source location: "python -m pip install pandas ipykernel numpy jupyter"

In [36]:
import pandas as pd
import numpy as np

In [37]:
play_attention_scores = pd.read_csv("../outputs_csv/play_attention_scores.csv")
pffScoutingData = pd.read_csv("../cleaned_csv/pffScoutingData_cleaned.csv")
plays = pd.read_csv("../cleaned_csv/plays_cleaned_cleaned.csv")
play_context = pd.read_csv("../outputs_csv/play_context.csv")
# The each_pass_rusher dataframe contains each pass rusher for each play.
each_pass_rusher = pd.read_csv("../outputs_csv/each_pass_rusher.csv")

In [38]:
play_attention_scores.rename(columns={"rusher_nflId": "nflId"}, inplace=True)

**VERY IMPORTANT!**

The base_filtered file below is not in the GitHub because it is too large. It is in a .zip file in the shared **Google Drive** folder inside the "Output Datasets" folder. The zip file is called "Datasets Not in Github (Michael).zip". 

After extracting the .zip file, drag the base_filtered.csv file into the outputs_csv folder.

In [39]:
# The base_filtered dataframe contains frame-by-frame data of all players on only relevant plays.
base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")



/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_89982/3775231977.py:2: DtypeWarning: Columns (0: foulNames, 1: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")


In [40]:
base_filtered['is_defense'] = base_filtered['pff_role'].isin(['Coverage', 'Pass Rush'])
base_filtered['team'] = np.where(
    base_filtered['is_defense'],
    base_filtered['defensiveTeam'],
    base_filtered['possessionTeam'],
)

base_filtered[['pff_role', 'possessionTeam', 'defensiveTeam', 'team', 'is_defense']].head()

,pff_role,possessionTeam,defensiveTeam,team,is_defense
0,Pass,TB,DAL,TB,False
1,Pass,TB,DAL,TB,False
2,Pass,TB,DAL,TB,False
3,Pass,TB,DAL,TB,False
4,Pass,TB,DAL,TB,False


In [41]:
pffScoutingData

,gameId,playId,nflId,pff_role,pff_positionLinedUp,pff_nflIdBlockedPlayer,pff_blockType,pff_backFieldBlock
0,2021090900,97,25511,Pass,QB,NaN,NaN,NaN
1,2021090900,97,35481,Pass Route,TE-L,NaN,NaN,NaN
2,2021090900,97,35634,Pass Route,LWR,NaN,NaN,NaN
3,2021090900,97,39985,Pass Route,HB-R,NaN,NaN,NaN
4,2021090900,97,40151,Pass Block,C,44955.0,SW,0.0
...,...,...,...,...,...,...,...,...
188249,2021110100,4433,52507,Pass Block,LT,43338.0,PP,0.0
188250,2021110100,4433,52546,Coverage,SCBoR,NaN,NaN,NaN
188251,2021110100,4433,52573,Pass Route,SLoWR,NaN,NaN,NaN
188252,2021110100,4433,52585,Pass Rush,LEO,NaN,NaN,NaN


Create dataframes that can be used for later.

In [42]:
# ball_snap_frames contains only frames when the ball is snapped
ball_snap_frames = base_filtered[base_filtered['event'] == 'ball_snap']

# base_filtered_pass_rushers is for pass rushers only, and it includes frames before the pass rush window unlike pass_rushers.
base_filtered_pass_rushers = base_filtered.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')

Information for LLM or AI prompting: Each combination of gameId and playId represents an individual play. Each combination of gameId, playId, and nflId represents an individual player on an indiviual play.

In [43]:
ball_snap_frames.columns

Index(['gameId', 'playId', 'season', 'week', 'gameDate', 'quarter', 'down',
       'yardsToGo', 'gameClock', 'play time', 'frameId', 'possessionTeam',
       'defensiveTeam', 'yardlineSide', 'yardlineNumber',
       'absoluteYardlineNumber', 'offenseFormation', 'offenseRB', 'offenseTE',
       'offenseWR', 'defendersInBox', 'defenseDL', 'defenseLB', 'defenseDB',
       'dropBackType', 'playAction', 'passCoverage', 'passCoverageType',
       'is_screen', 'is_rpo', 'is_qb_spike', 'is_gravity_candidate_base',
       'preSnapHomeScore', 'preSnapVisitorScore', 'passResult', 'penaltyYards',
       'prePenaltyPlayResult', 'playResult', 'foulNames', 'foulIds', 'nflId',
       'pff_role', 'pff_positionLinedUp', 'pff_nflIdBlockedPlayer',
       'pff_blockType', 'pff_backFieldBlock', 'height', 'weight',
       'officialPosition', 'displayName', 'jerseyNumber', 'playDirection', 'x',
       'y', 's', 'a', 'dis', 'o', 'dir', 'event', 'frameIdEndWindow',
       'is_defense', 'team'],
      dtype='str

Feature-engineer variables for how far the pass rusher is from the center at the ball_snap instance (x distance, y distance, and total distance).

In [44]:


# filter ball_snap_frames to only include ball snap frames of pass rushers in each_pass_rusher df
ball_snaps_pass_rushers = ball_snap_frames.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')
ball_snaps_pass_rushers

# ball_snap_frames to only include centers
ball_snaps_centers = ball_snap_frames[ball_snap_frames['pff_positionLinedUp'] == 'C']
ball_snaps_centers

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,y,s,a,dis,o,dir,event,frameIdEndWindow,is_defense,team
149,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,24.02,0.51,1.98,0.06,48.28,308.34,ball_snap,36,False,TB
1418,2021090900,137,2021,1,09/09/2021,1,1,10,13:18,28:10.500,...,24.06,0.92,0.88,0.09,279.04,170.92,ball_snap,31,False,DAL
2046,2021090900,187,2021,1,09/09/2021,1,2,6,12:23,29:15.500,...,26.46,0.54,1.41,0.06,284.80,121.34,ball_snap,27,False,DAL
2829,2021090900,282,2021,1,09/09/2021,1,1,10,9:56,31:52.100,...,30.07,0.58,0.87,0.06,265.80,123.15,ball_snap,36,False,DAL
3506,2021090900,349,2021,1,09/09/2021,1,3,15,9:46,34:05.600,...,30.11,0.82,2.07,0.08,276.68,94.61,ball_snap,32,False,DAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5163669,2021110100,4310,2021,8,11/01/2021,4,3,8,1:56,15:52.900,...,30.05,0.39,1.38,0.03,282.30,91.81,ball_snap,37,False,KC
5164113,2021110100,4363,2021,8,11/01/2021,4,1,10,1:07,18:41.000,...,29.92,0.55,1.39,0.05,75.80,288.03,ball_snap,37,False,NYG
5164927,2021110100,4392,2021,8,11/01/2021,4,2,7,1:01,19:17.900,...,23.77,0.54,1.54,0.06,79.79,278.44,ball_snap,37,False,NYG
5165696,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,23.78,0.62,1.08,0.06,81.31,273.62,ball_snap,32,False,NYG


In [45]:
# calculate the difference in x, the difference in y, and the euclidean distance of each pass rusher to the center at the moment of the ball snap on each play.
temp = ball_snaps_pass_rushers.merge(ball_snaps_centers[['gameId', 'playId', 'x', 'y']], on=['gameId', 'playId'], how='inner', suffixes=('', '_center'))
temp['diff_x'] = temp['x'] - temp['x_center']
temp['diff_y'] = temp['y'] - temp['y_center']
temp['euclidean_distance'] = np.sqrt(temp['diff_x']**2 + temp['diff_y']**2)
temp

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,dir,event,frameIdEndWindow,is_defense,team,x_center,y_center,diff_x,diff_y,euclidean_distance
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,288.76,ball_snap,36,True,DAL,42.10,24.02,1.20,-5.13,5.268482
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,247.75,ball_snap,36,True,DAL,42.10,24.02,1.80,8.61,8.796141
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,288.42,ball_snap,36,True,DAL,42.10,24.02,1.25,1.16,1.705315
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,316.78,ball_snap,36,True,DAL,42.10,24.02,1.58,-2.09,2.620019
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,345.84,ball_snap,36,True,DAL,42.10,24.02,1.60,2.65,3.095561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,249.67,ball_snap,32,True,KC,29.07,23.78,1.22,4.86,5.010788
30967,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,271.87,ball_snap,37,True,KC,29.15,23.72,1.16,6.07,6.179846
30968,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,251.87,ball_snap,37,True,KC,29.15,23.72,1.04,-2.90,3.080844
30969,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,283.80,ball_snap,37,True,KC,29.15,23.72,1.30,2.93,3.205448


In [46]:
# create a training dataset for each pass rusher on each play.
training_dataset = each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates()
training_dataset = training_dataset.merge(temp[['gameId', 'playId', 'nflId', 'diff_x', 'diff_y', 'euclidean_distance']], on=['gameId', 'playId', 'nflId'], how='left')




Feature-engineer other variables and add them to the `training_dataset` dataframe.

PFF Position Lined Up

In [47]:
temp = ball_snaps_pass_rushers.copy()
temp['pff_positionLinedUp'].value_counts()


# group RE and LE. group REO and LEO. group ROLB and LOLB. group DRT and DLT. group NLT, NRT, and NT.
def group_positions(position):
    if position in ['RE', 'LE']:
        return 'End'
    elif position in ['REO', 'LEO']:
        return 'End Outside'
    elif position in ['ROLB', 'LOLB']:
        return 'OLB'
    elif position in ['DRT', 'DLT']:
        return 'DT'
    elif position in ['NLT', 'NRT', 'NT']:
        return 'NT'
    elif position in ['RILB', 'LILB', 'MLB']:
        return 'ILB'
    elif position in ['RLB', 'LLB']:
        return 'LB'
    else:
        return "Secondary"
    
temp['grouped_position'] = temp['pff_positionLinedUp'].apply(group_positions)

# merge play_attention_scores with ball_snaps_pass_rushers on gameId, playId, and nflId to get the grouped for each pass rusher in the play_attention_scores dataframe.
scores_temp = play_attention_scores.merge(
    temp[['gameId', 'playId', 'nflId', 'grouped_position']],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)
scores_temp.groupby('grouped_position')['avg_attention_score'].mean().sort_values(ascending=False)

grouped_position
DT             1.440668
End            1.426816
NT             1.423070
End Outside    1.008230
OLB            0.885306
ILB            0.864821
LB             0.541355
Secondary      0.309285
Name: avg_attention_score, dtype: float64

In [48]:
# group "DT", "NT", and "End" as "DI". group "End Outside" and "OLB" as "Edge". group "LB" and "Secondary" as "Other"

# create function
def group_positions_final(position):
    if position in ['DT', 'NT', 'End']:
        return 'DI'
    elif position in ['End Outside', 'OLB']:
        return 'Edge'
    elif position in ['LB', 'Secondary']:
        return 'Other'
    else:
        return position

temp['grouped_position'] = temp['grouped_position'].apply(group_positions_final)

# do one-hot encoding on temp['grouped_position'] column and merge it back to the training_dataset on gameId, playId, and nflId.

one_hot = pd.get_dummies(temp['grouped_position'], prefix='pos')
# convert from boolean to integer
one_hot = one_hot.astype(int)
temp = pd.concat([temp[['gameId', 'playId', 'nflId']], one_hot], axis=1)
training_dataset = training_dataset.merge(
    temp,
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)




Number of rushers who are rushing

In [49]:
# Count pass rushers by play. playId can repeat across games, so group by both gameId and playId.
num_blitzing_rushers_by_play = (
    each_pass_rusher[['gameId', 'playId', 'nflId']]
    .drop_duplicates()
    .groupby(['gameId', 'playId'], as_index=False)
    .agg(num_blitzing_rushers=('nflId', 'count'))
)

training_dataset = training_dataset.merge(
    num_blitzing_rushers_by_play,
    on=['gameId', 'playId'],
    how='left',
    validate='many_to_one',
)

training_dataset

,gameId,playId,nflId,diff_x,diff_y,euclidean_distance,pos_DI,pos_Edge,pos_ILB,pos_Other,num_blitzing_rushers
0,2021090900,97,41263,1.20,-5.13,5.268482,0,1,0,0,5
1,2021090900,97,42403,1.80,8.61,8.796141,0,1,0,0,5
2,2021090900,97,44955,1.25,1.16,1.705315,1,0,0,0,5
3,2021090900,97,53441,1.58,-2.09,2.620019,0,0,1,0,5
4,2021090900,97,53504,1.60,2.65,3.095561,1,0,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,52585,1.22,4.86,5.010788,0,1,0,0,5
30967,2021110100,4433,42406,1.16,6.07,6.179846,0,1,0,0,4
30968,2021110100,4433,43326,1.04,-2.90,3.080844,1,0,0,0,4
30969,2021110100,4433,43338,1.30,2.93,3.205448,1,0,0,0,4


Number of "expected rushers" - defensive players lined up near the LOS (excluding CBs)

In [50]:
## number of defensive players lined up within 2.5 yards in x distance from the center. Do not include any type of cornerbacks.

center_locations = (
    ball_snaps_centers[['gameId', 'playId', 'x']]
    .drop_duplicates(subset=['gameId', 'playId'])
    .rename(columns={'x': 'x_center'})
)

ball_snap_defenders = ball_snap_frames.merge(
    center_locations,
    on=['gameId', 'playId'],
    how='inner',
    validate='many_to_one',
)

non_cornerback_defender_near_center = (
    (ball_snap_defenders['team'] == ball_snap_defenders['defensiveTeam'])
    & ((ball_snap_defenders['x'] - ball_snap_defenders['x_center']).abs() <= 2.5)
    & (~ball_snap_defenders['pff_positionLinedUp'].str.contains('CB', case=False, na=False))
)

defenders_on_los = (
    ball_snap_defenders[non_cornerback_defender_near_center]
    .groupby(['gameId', 'playId'], as_index=False)
    .agg(numExpectedRushers=('nflId', 'nunique'))
)

training_dataset = training_dataset.merge(
    defenders_on_los,
    on=['gameId', 'playId'],
    how='left',
    validate='many_to_one',
)

training_dataset['numExpectedRushers'] = training_dataset['numExpectedRushers'].fillna(0).astype(int)

# blitzers vs expected = number of blitzing rushers - number of defenders near the line of scrimmage
training_dataset['blitzers_vs_expected'] = training_dataset['num_blitzing_rushers'] - training_dataset['numExpectedRushers']

In [51]:
# merge training_dataset with the above "ball_snap_defenders[non_cornerback_defender_near_center]" to get a boolean feature to see if the pass rusher is lined up on the line of scrimmage. Use gameId, playId, and nflId. If not, then it is false.
pass_rushers_on_los = (
    ball_snap_defenders.loc[
        non_cornerback_defender_near_center,
        ['gameId', 'playId', 'nflId'],
    ]
    .drop_duplicates()
    .assign(isExpectedRusher=True)
)

training_dataset = training_dataset.merge(
    pass_rushers_on_los,
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)

training_dataset['isExpectedRusher'] = training_dataset['isExpectedRusher'].fillna(False)

# convert above column to 0s and 1s
training_dataset['isExpectedRusher'] = training_dataset['isExpectedRusher'].astype(int)

In [52]:
training_dataset['isExpectedRusher'].value_counts()

isExpectedRusher
1    29492
0     1479
Name: count, dtype: int64

Number of rushers on the play who weren't "expected rushers"

In [53]:
# Count pass rushers by play who rushed from away from the line of scrimmage.
# Cornerbacks who rush are counted as not near the LOS, matching pass_rusher_near_los.
pass_rusher_not_near_los = training_dataset['isExpectedRusher'] == 0

non_los_pass_rushers_by_play = (
    training_dataset.loc[pass_rusher_not_near_los, ['gameId', 'playId', 'nflId']]
    .drop_duplicates()
    .groupby(['gameId', 'playId'], as_index=False)
    .agg(num_non_expected_rushers=('nflId', 'count'))
)

training_dataset = training_dataset.merge(
    non_los_pass_rushers_by_play,
    on=['gameId', 'playId'],
    how='left',
    validate='many_to_one',
)

training_dataset['num_non_expected_rushers'] = training_dataset['num_non_expected_rushers'].fillna(0).astype(int)
training_dataset['has_non_expected_rusher'] = (training_dataset['num_non_expected_rushers'] > 0).astype(int)
training_dataset['potential_rushers'] = training_dataset['numExpectedRushers'] + training_dataset['num_non_expected_rushers']


Number of pass rushers who are on the rusher's side of the center

In [54]:
# calculate the number and proportion of other pass rushers who are on each rusher's side of the center at the moment of the ball snap on each play. 
pass_rusher_sides = training_dataset[['gameId', 'playId', 'nflId', 'diff_y', 'num_blitzing_rushers']].drop_duplicates().copy()
pass_rusher_sides['rusher_side_of_center'] = np.select(
    [pass_rusher_sides['diff_y'] > 0, pass_rusher_sides['diff_y'] < 0],
    [1, -1],
    default=0,
)

pass_rushers_by_side = (
    pass_rusher_sides
    .groupby(['gameId', 'playId', 'rusher_side_of_center'], as_index=False)
    .agg(numPassRushersOnRusherSide=('nflId', 'nunique'))
)

other_pass_rusher_side_features = pass_rusher_sides.merge(
    pass_rushers_by_side,
    on=['gameId', 'playId', 'rusher_side_of_center'],
    how='left',
    validate='many_to_one',
)

other_pass_rusher_side_features['numOtherPassRushersOnRusherSide'] = (
    other_pass_rusher_side_features['numPassRushersOnRusherSide'] - 1
).clip(lower=0).astype(int)

other_pass_rushers_by_play = other_pass_rusher_side_features['num_blitzing_rushers'] - 1
other_pass_rusher_side_features['propOtherPassRushersOnRusherSide'] = np.where(
    other_pass_rushers_by_play > 0,
    other_pass_rusher_side_features['numOtherPassRushersOnRusherSide'] / other_pass_rushers_by_play,
    0,
)

training_dataset = training_dataset.merge(
    other_pass_rusher_side_features[
        [
            'gameId',
            'playId',
            'nflId',
            'numOtherPassRushersOnRusherSide',
            'propOtherPassRushersOnRusherSide',
        ]
    ],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)

training_dataset['numOtherPassRushersOnRusherSide'] = training_dataset['numOtherPassRushersOnRusherSide'].fillna(0).astype(int)
training_dataset['propOtherPassRushersOnRusherSide'] = training_dataset['propOtherPassRushersOnRusherSide'].fillna(0)


In [55]:
# calculate the number and proportion of expected pass rushers (players lined up near the LOS and not a CB) who are on the rusher's side of the center.
center_y_locations = (
    ball_snaps_centers[['gameId', 'playId', 'y']]
    .drop_duplicates(subset=['gameId', 'playId'])
    .rename(columns={'y': 'y_center'})
)

expected_rusher_sides = (
    ball_snap_defenders.loc[
        non_cornerback_defender_near_center,
        ['gameId', 'playId', 'nflId', 'y'],
    ]
    .drop_duplicates()
    .merge(
        center_y_locations,
        on=['gameId', 'playId'],
        how='inner',
        validate='many_to_one',
    )
)

expected_rusher_sides['rusher_side_of_center'] = np.select(
    [expected_rusher_sides['y'] > expected_rusher_sides['y_center'], expected_rusher_sides['y'] < expected_rusher_sides['y_center']],
    [1, -1],
    default=0,
)

expected_rushers_by_side = (
    expected_rusher_sides
    .groupby(['gameId', 'playId', 'rusher_side_of_center'], as_index=False)
    .agg(numExpectedRushersOnRusherSide=('nflId', 'nunique'))
)

pass_rusher_sides = training_dataset[['gameId', 'playId', 'nflId', 'diff_y', 'numExpectedRushers', 'isExpectedRusher']].drop_duplicates().copy()
pass_rusher_sides['rusher_side_of_center'] = np.select(
    [pass_rusher_sides['diff_y'] > 0, pass_rusher_sides['diff_y'] < 0],
    [1, -1],
    default=0,
)

expected_rusher_side_features = pass_rusher_sides.merge(
    expected_rushers_by_side,
    on=['gameId', 'playId', 'rusher_side_of_center'],
    how='left',
    validate='many_to_one',
)

expected_rusher_side_features['numExpectedRushersOnRusherSide'] = expected_rusher_side_features['numExpectedRushersOnRusherSide'].fillna(0).astype(int)
other_expected_rushers_on_side = (
    expected_rusher_side_features['numExpectedRushersOnRusherSide'] - expected_rusher_side_features['isExpectedRusher']
).clip(lower=0)
other_expected_rushers_by_play = (
    expected_rusher_side_features['numExpectedRushers'] - expected_rusher_side_features['isExpectedRusher']
).clip(lower=0)
expected_rusher_side_features['numExpectedRushersOnRusherSide'] = other_expected_rushers_on_side.astype(int)

expected_rusher_side_features['propExpectedRushersOnRusherSide'] = np.where(
    other_expected_rushers_by_play > 0,
    other_expected_rushers_on_side / other_expected_rushers_by_play,
    0,
)

training_dataset = training_dataset.merge(
    expected_rusher_side_features[
        [
            'gameId',
            'playId',
            'nflId',
            'numExpectedRushersOnRusherSide',
            'propExpectedRushersOnRusherSide',
        ]
    ],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)

training_dataset['numExpectedRushersOnRusherSide'] = training_dataset['numExpectedRushersOnRusherSide'].fillna(0).astype(int)
training_dataset['propExpectedRushersOnRusherSide'] = training_dataset['propExpectedRushersOnRusherSide'].fillna(0)

In [56]:
# # test for correctness

# temp = ball_snap_defenders.loc[non_cornerback_defender_near_center,
#         ['gameId', 'playId', 'nflId', 'y'],
#     ].drop_duplicates().merge(
#         center_y_locations,
#         on=['gameId', 'playId'],
#         how='inner',
#         validate='many_to_one',
#     )
# print(temp[(temp['gameId'] == 2021090900) & (temp['playId'] == 1563) ])

# training_dataset[(training_dataset['gameId'] == 2021090900) & (training_dataset['playId'] == 1563) ]


Down variable: Feature-engineer down variable using one-hot encoding. We group downs 3 and 4

In [57]:
ball_snaps_pass_rushers['down_grouped'] = ball_snaps_pass_rushers['down'].apply(lambda x: '3-4' if x in [3, 4] else str(x))

# do one-hot encoding for down using ball_snaps_pass_rushers['down'] and merge to training_dataset. Use "down_grouped".  
down_dummies = pd.get_dummies(ball_snaps_pass_rushers['down_grouped'], prefix='down')
# convert boolean to integer
down_dummies = down_dummies.astype(int)
ball_snaps_pass_rushers_with_dummies = pd.concat([ball_snaps_pass_rushers[['gameId', 'playId', 'nflId']], down_dummies], axis=1)
training_dataset = training_dataset.merge(
    ball_snaps_pass_rushers_with_dummies,
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)   

In [58]:
training_dataset.iloc[:,0:20]

,gameId,playId,nflId,diff_x,diff_y,euclidean_distance,pos_DI,pos_Edge,pos_ILB,pos_Other,num_blitzing_rushers,numExpectedRushers,blitzers_vs_expected,isExpectedRusher,num_non_expected_rushers,has_non_expected_rusher,potential_rushers,numOtherPassRushersOnRusherSide,propOtherPassRushersOnRusherSide,numExpectedRushersOnRusherSide
0,2021090900,97,41263,1.20,-5.13,5.268482,0,1,0,0,5,5,0,1,0,0,5,1,0.250000,1
1,2021090900,97,42403,1.80,8.61,8.796141,0,1,0,0,5,5,0,1,0,0,5,2,0.500000,2
2,2021090900,97,44955,1.25,1.16,1.705315,1,0,0,0,5,5,0,1,0,0,5,2,0.500000,2
3,2021090900,97,53441,1.58,-2.09,2.620019,0,0,1,0,5,5,0,1,0,0,5,1,0.250000,1
4,2021090900,97,53504,1.60,2.65,3.095561,1,0,0,0,5,5,0,1,0,0,5,2,0.500000,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,52585,1.22,4.86,5.010788,0,1,0,0,5,4,1,1,1,1,5,1,0.250000,1
30967,2021110100,4433,42406,1.16,6.07,6.179846,0,1,0,0,4,5,-1,1,0,0,5,1,0.333333,1
30968,2021110100,4433,43326,1.04,-2.90,3.080844,1,0,0,0,4,5,-1,1,0,0,5,1,0.333333,2
30969,2021110100,4433,43338,1.30,2.93,3.205448,1,0,0,0,4,5,-1,1,0,0,5,1,0.333333,1


Add additional features.

In [59]:
# add more features

# Feature-engineer relativeYardLine from the raw yardlineNumber and yardlineSide.
# yardlineNumber is relative to yardlineSide, so flip it when the ball is on the defensive team's side.
play_context_features = ball_snaps_pass_rushers[
    [
        'gameId',
        'playId',
        'nflId',
        'possessionTeam',
        'yardlineSide',
        'yardlineNumber',
        'yardsToGo',
        'defendersInBox',
        'quarter',
    ]
].copy()

play_context_features['relativeYardLine'] = play_context_features['yardlineNumber'].where(
    play_context_features['yardlineSide'].eq(play_context_features['possessionTeam']),
    100 - play_context_features['yardlineNumber'],
)

# add relativeYardLine, yardsToGo, defendersInBox, and quarter to training_dataset.
training_dataset = training_dataset.merge(
    play_context_features[
        ['gameId', 'playId', 'nflId', 'relativeYardLine', 'yardsToGo', 'defendersInBox', 'quarter']
    ],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)


In [60]:
# feature-engineer a variable for half

training_dataset['half'] = np.where(training_dataset['quarter'] <= 2, 1, 2)

# feature-engineer variables for seconds left in half and seconds left in game.
clock_features = ball_snaps_pass_rushers[['gameId', 'playId', 'nflId', 'quarter', 'gameClock']].copy()
clock_parts = clock_features['gameClock'].str.split(':', expand=True).astype(int)
clock_features['secondsLeftInQuarter'] = clock_parts[0] * 60 + clock_parts[1]
clock_features['secondsLeftInHalf'] = np.where(
    clock_features['quarter'].isin([1, 3]),
    clock_features['secondsLeftInQuarter'] + 15 * 60,
    clock_features['secondsLeftInQuarter'],
)
clock_features['secondsLeftInGame'] = (
    (4 - clock_features['quarter']).clip(lower=0) * 15 * 60
    + clock_features['secondsLeftInQuarter']
)

training_dataset = training_dataset.merge(
    clock_features[['gameId', 'playId', 'nflId', 'secondsLeftInHalf', 'secondsLeftInGame']],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)


Feature-engineer a variable for point difference at time of snap (relative to possession team)

In [61]:
original_games = pd.read_csv("../cleaned_csv/games_cleaned.csv")
original_games

score_features = ball_snaps_pass_rushers[
    [
        'gameId',
        'playId',
        'nflId',
        'possessionTeam',
        'preSnapHomeScore',
        'preSnapVisitorScore',
    ]
].copy()

score_features = score_features.merge(
    original_games[['gameId', 'homeTeamAbbr', 'visitorTeamAbbr']],
    on='gameId',
    how='left',
    validate='many_to_one',
)

possession_is_home = score_features['possessionTeam'].eq(score_features['homeTeamAbbr'])
possession_is_visitor = score_features['possessionTeam'].eq(score_features['visitorTeamAbbr'])

score_features['winningBy'] = np.select(
    [possession_is_home, possession_is_visitor],
    [
        score_features['preSnapHomeScore'] - score_features['preSnapVisitorScore'],
        score_features['preSnapVisitorScore'] - score_features['preSnapHomeScore'],
    ],
    default=np.nan,
)

training_dataset = training_dataset.merge(
    score_features[['gameId', 'playId', 'nflId', 'winningBy']],
    on=['gameId', 'playId', 'nflId'],
    how='left',
    validate='one_to_one',
)


In [62]:
ball_snaps_pass_rushers.columns

Index(['gameId', 'playId', 'season', 'week', 'gameDate', 'quarter', 'down',
       'yardsToGo', 'gameClock', 'play time', 'frameId', 'possessionTeam',
       'defensiveTeam', 'yardlineSide', 'yardlineNumber',
       'absoluteYardlineNumber', 'offenseFormation', 'offenseRB', 'offenseTE',
       'offenseWR', 'defendersInBox', 'defenseDL', 'defenseLB', 'defenseDB',
       'dropBackType', 'playAction', 'passCoverage', 'passCoverageType',
       'is_screen', 'is_rpo', 'is_qb_spike', 'is_gravity_candidate_base',
       'preSnapHomeScore', 'preSnapVisitorScore', 'passResult', 'penaltyYards',
       'prePenaltyPlayResult', 'playResult', 'foulNames', 'foulIds', 'nflId',
       'pff_role', 'pff_positionLinedUp', 'pff_nflIdBlockedPlayer',
       'pff_blockType', 'pff_backFieldBlock', 'height', 'weight',
       'officialPosition', 'displayName', 'jerseyNumber', 'playDirection', 'x',
       'y', 's', 'a', 'dis', 'o', 'dir', 'event', 'frameIdEndWindow',
       'is_defense', 'team', 'down_grouped'],


In [63]:


training_dataset.isna().sum()

gameId                              0
playId                              0
nflId                               0
diff_x                              0
diff_y                              0
euclidean_distance                  0
pos_DI                              0
pos_Edge                            0
pos_ILB                             0
pos_Other                           0
num_blitzing_rushers                0
numExpectedRushers                  0
blitzers_vs_expected                0
isExpectedRusher                    0
num_non_expected_rushers            0
has_non_expected_rusher             0
potential_rushers                   0
numOtherPassRushersOnRusherSide     0
propOtherPassRushersOnRusherSide    0
numExpectedRushersOnRusherSide      0
propExpectedRushersOnRusherSide     0
down_1                              0
down_2                              0
down_3-4                            0
relativeYardLine                    0
yardsToGo                           0
defendersInB

### Yardline number check on opening drive plays (Made by LLM)

`plays.csv` does not include a drive id, so this treats the first listed play after a possession-team change within each game as the opening observed play of that possession segment.

In [64]:
# original_plays = pd.read_csv("../../datasets/plays.csv")
# original_plays

# opening_drive_plays = original_plays.sort_values(['gameId', 'playId']).copy()
# opening_drive_plays['is_opening_observed_drive_play'] = (
#     opening_drive_plays.groupby('gameId')['possessionTeam'].transform(lambda s: s.ne(s.shift()))
# )
# opening_drive_plays = opening_drive_plays[opening_drive_plays['is_opening_observed_drive_play']].copy()

# opening_drive_plays['yardline_side_is_possession'] = opening_drive_plays['yardlineSide'].eq(
#     opening_drive_plays['possessionTeam']
# )
# opening_drive_plays['possession_relative_yardline'] = opening_drive_plays['yardlineNumber'].where(
#     opening_drive_plays['yardline_side_is_possession'],
#     100 - opening_drive_plays['yardlineNumber'],
# )

# opening_drive_yardline_summary = (
#     opening_drive_plays
#     .groupby('yardline_side_is_possession')[['yardlineNumber', 'possession_relative_yardline']]
#     .describe()
# )

# opening_drive_opponent_side_examples = opening_drive_plays.loc[
#     ~opening_drive_plays['yardline_side_is_possession'],
#     [
#         'gameId',
#         'playId',
#         'quarter',
#         'gameClock',
#         'possessionTeam',
#         'defensiveTeam',
#         'yardlineSide',
#         'yardlineNumber',
#         'possession_relative_yardline',
#         'absoluteYardlineNumber',
#         'playDescription',
#     ],
# ].head(20)

# print(f"Opening observed drive plays: {len(opening_drive_plays):,}")
# display(opening_drive_plays['yardline_side_is_possession'].value_counts(dropna=False))
# display(opening_drive_yardline_summary)
# display(opening_drive_opponent_side_examples)


**Conclusion:** `yardlineNumber` is relative to `yardlineSide`, not always relative to the possession team. When `yardlineSide == possessionTeam`, it is the offense's own marked yard line. When `yardlineSide != possessionTeam`, the offense is already on the opponent's side, so the possession-relative field position is `100 - yardlineNumber`.

### Constructing the Model

In [65]:
training_dataset = training_dataset.merge(play_attention_scores, on=['gameId', 'playId', 'nflId'], how='inner')
training_dataset


,gameId,playId,nflId,diff_x,diff_y,euclidean_distance,pos_DI,pos_Edge,pos_ILB,pos_Other,...,relativeYardLine,yardsToGo,defendersInBox,quarter,half,secondsLeftInHalf,secondsLeftInGame,winningBy,avg_attention_score,rusher_name
0,2021090900,97,41263,1.20,-5.13,5.268482,0,1,0,0,...,33,2,6.0,1,1,1713,3513,0.0,0.461538,Demarcus Lawrence
1,2021090900,97,42403,1.80,8.61,8.796141,0,1,0,0,...,33,2,6.0,1,1,1713,3513,0.0,0.615385,Randy Gregory
2,2021090900,97,44955,1.25,1.16,1.705315,1,0,0,0,...,33,2,6.0,1,1,1713,3513,0.0,1.307692,Carlos Watkins
3,2021090900,97,53441,1.58,-2.09,2.620019,0,0,1,0,...,33,2,6.0,1,1,1713,3513,0.0,1.000000,Micah Parsons
4,2021090900,97,53504,1.60,2.65,3.095561,1,0,0,0,...,33,2,6.0,1,1,1713,3513,0.0,0.615385,Osa Odighizuwa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,52585,1.22,4.86,5.010788,0,1,0,0,...,20,15,5.0,4,2,39,39,-3.0,0.904762,Michael Danna
30967,2021110100,4433,42406,1.16,6.07,6.179846,0,1,0,0,...,20,15,6.0,4,2,35,35,-3.0,0.923077,Frank Clark
30968,2021110100,4433,43326,1.04,-2.90,3.080844,1,0,0,0,...,20,15,6.0,4,2,35,35,-3.0,1.500000,Chris Jones
30969,2021110100,4433,43338,1.30,2.93,3.205448,1,0,0,0,...,20,15,6.0,4,2,35,35,-3.0,1.346154,Jarran Reed


In [66]:
# let x be all features except first three and last two columns
X = training_dataset.drop(columns=['gameId', 'playId', 'nflId', 'avg_attention_score', 'rusher_name'])
y = training_dataset['avg_attention_score']


from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X, y)
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

# r2score
from sklearn.metrics import r2_score
y_pred = model.predict(X)
print("R^2 Score:", r2_score(y, y_pred))

Coefficients: [ 6.43075584e-04  2.06817239e-04 -9.48632371e-02  1.20994532e-01
 -4.60277510e-02 -1.21754454e-01  4.67876735e-02 -1.11826867e-01
 -7.54867362e-02 -3.63401307e-02  4.99471569e-01  7.84504928e-02
  4.03880196e-02  2.96375655e-03  2.45990918e-02 -3.77852159e-01
  8.45066307e-02 -2.15114353e-01  3.65508435e-03 -1.29724952e-02
  9.31741088e-03 -5.04391904e-04  2.11132494e-03  1.08686541e-02
 -5.20617429e-03  8.25674147e-10 -3.60927404e-06 -5.09548756e-06
 -9.12647171e-05]
Intercept: 1.8575605571897813
R^2 Score: 0.38687971620852457


Preliminary multiple regression test (Not Our Final Model)

In [67]:
# # Calculate gravity for Myles Garrett based on this regression model.
# myles_garrett_temp = training_dataset[training_dataset['rusher_name'] == 'Myles Garrett']
# X = myles_garrett_temp.drop(columns=['gameId', 'playId', 'nflId', 'avg_attention_score', 'rusher_name'])

# myles_garrett_temp['predicted_attention_score'] = model.predict(X)
# myles_garrett_temp['gravity_score'] = myles_garrett_temp['avg_attention_score'] - myles_garrett_temp['predicted_attention_score']
# myles_garrett_avg_gravity = myles_garrett_temp['gravity_score'].mean()
# print("Myles Garrett's average gravity:", myles_garrett_avg_gravity)  

# # Calculate gravity for Aaron Donald based on this regression model.
# aaron_donald_temp = training_dataset[training_dataset['rusher_name'] == 'Aaron Donald']
# X = aaron_donald_temp.drop(columns=['gameId', 'playId', 'nflId', 'avg_attention_score', 'rusher_name'])
# aaron_donald_temp['predicted_attention_score'] = model.predict(X)
# aaron_donald_temp['gravity_score'] = aaron_donald_temp['avg_attention_score'] - aaron_donald_temp['predicted_attention_score']
# aaron_donald_avg_gravity = aaron_donald_temp['gravity_score'].mean()
# print("Aaron Donald's average gravity:", aaron_donald_avg_gravity)  

# # Calculate gravity for Justin Hollins based on this regression model.
# justin_hollins_temp = training_dataset[training_dataset['rusher_name'] == 'Justin Hollins']
# X = justin_hollins_temp.drop(columns=['gameId', 'playId', 'nflId', 'avg_attention_score', 'rusher_name'])
# justin_hollins_temp['predicted_attention_score'] = model.predict(X)
# justin_hollins_temp['gravity_score'] = justin_hollins_temp['avg_attention_score'] - justin_hollins_temp['predicted_attention_score']
# justin_hollins_avg_gravity = justin_hollins_temp['gravity_score'].mean()
# print("Justin Hollins's average gravity:", justin_hollins_avg_gravity)


In [ ]:
# output training datasets


y_output = training_dataset[['avg_attention_score', 'rusher_name']]
y_output

X.to_csv("../outputs_csv/X_train.csv", index=False)
y_output.to_csv("../outputs_csv/y_train.csv", index=False)


## Neural Net + Softmax

In [ ]:
training_dataset_nn = training_dataset.copy().reset_index(drop=True)

training_dataset_nn = training_dataset_nn[
    training_dataset_nn['isExpectedRusher'] == 1
]

training_dataset_nn = training_dataset_nn.groupby(
    ['gameId', 'playId', 'nflId', 'rusher_name'],
    as_index=False
).mean()

In [ ]:
from sklearn.preprocessing import StandardScaler
X_df = training_dataset_nn.drop(columns=[
    'gameId','playId','nflId','avg_attention_score','rusher_name'
])

X_df = X_df.astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)
X = X_scaled.astype(np.float32)

play_ids = training_dataset_nn['playId'].values


training_dataset_nn['y_norm'] = training_dataset_nn.groupby(['gameId', 'playId'])[
    'avg_attention_score'
].transform(lambda x: x / x.sum())

y = training_dataset_nn['y_norm'].values

In [ ]:
import torch
import torch.nn as nn

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)



In [ ]:
class AttentionNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()

In [ ]:
model = AttentionNet(X.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
unique_plays = np.unique(play_ids)

for epoch in range(30):
    total_loss = 0
    
    optimizer.zero_grad()


    for play in unique_plays:
        idx = (play_ids == play)

        X_play = X_tensor[idx]
        y_play = y_tensor[idx]

        scores = model(X_play)
        probs = torch.softmax(scores, dim=0)

        loss = -torch.sum(y_play * torch.log(probs + 1e-9)) / len(y_play)

        loss.backward()

        total_loss += loss.item()
    
    optimizer.step()

    if epoch % 5 == 0:
        avg_loss = total_loss / len(unique_plays)
        print(f"Epoch {epoch}, Average Loss: {avg_loss:.4f}")

In [ ]:
row = training_dataset_nn.iloc[0]

gameId = row['gameId']
playId = row['playId']


play_df = training_dataset_nn[
    (training_dataset_nn['gameId'] == gameId) &
    (training_dataset_nn['playId'] == playId)
]

feature_cols = X_df.columns

X_play_df = play_df[feature_cols]

X_play = X_play_df.astype(float).values
X_play = scaler.transform(X_play)

X_play_tensor = torch.tensor(X_play, dtype=torch.float32)

scores = model(X_play_tensor)
probs = torch.softmax(scores, dim=0).detach().numpy()

play_df = play_df.copy()
play_df['predicted_attention'] = probs


play_df = play_df.copy()
play_df['predicted_attention'] = probs

play_df['gravity'] = play_df['y_norm'] - play_df['predicted_attention']

print(
    play_df[['rusher_name','y_norm','predicted_attention','gravity']]
    .sort_values(by='gravity', ascending=False)
)

